# Classic examples

Verification of well-known nets from `petrilab.petri.examples`: producer/consumer, dining philosophers, mutual exclusion and a traffic crossing. We combine the reachability analysis (`analysis`), place invariants (`invariants`) and the SNAKES bridge.

```mermaid
flowchart LR
    ProdIdle((ProdIdle 1)) --> produce[produce] --> ProdIdle
    Free((Free N)) --> produce --> Buffer((Buffer))
    Buffer --> consume[consume] --> Free
    ConsIdle((ConsIdle 1)) --> consume --> ConsIdle
```

In [1]:
from petrilab.petri import analysis, examples
from petrilab.petri.invariants import is_place_invariant, place_invariants, weighted_sum
from petrilab.petri.snakes_bridge import explore, to_snakes


def space_of(net):
    return explore(to_snakes(net), token_cap=50)


n = examples.producer_consumer(2)
s = space_of(n)
print(
    "bounded:", analysis.is_bounded(s), "max Buffer:", analysis.max_tokens(s, "Buffer")
)
print(
    "deadlock free:",
    analysis.is_deadlock_free(s),
    "live:",
    analysis.is_live(s, set(n.transitions)),
)
print("states:", len(s))
print("Free+Buffer invariant:", is_place_invariant(n, {"Free": 1, "Buffer": 1}))

bounded: True max Buffer: 2
deadlock free: True live: True
states: 3
Free+Buffer invariant: True


The bounded buffer never exceeds its capacity of 2, the net is live and deadlock-free, and `Free + Buffer` is constant. The 3 states correspond to buffer fill levels 0, 1, 2.

In [2]:
for cap in range(1, 6):
    g = examples.producer_consumer(cap).reachable()
    print(cap, len(g), all(succ for succ in g.values()))

1 2 True
2 3 True
3 4 True
4 5 True
5 6 True


For any capacity `c` there are `c + 1` states and no deadlock.

In [3]:
n = examples.dining_philosophers(2)
s = space_of(n)
for key in analysis.dead_markings(s):
    print({p: len(v) for p, v in s.markings[key].items() if v})
print("live:", analysis.is_live(s, set(n.transitions)))

{'HasLeft0': 1, 'HasLeft1': 1}
live: False


The naive protocol (take left fork, then right fork) has a deadlock: each philosopher holds their left fork (`HasLeft0`, `HasLeft1`) and waits for the other.

In [4]:
n = examples.dining_philosophers(2)
m = n.initial_marking
for t in ("takeLeft0", "takeLeft1"):
    m = n.fire(m, t)
print(n.marking_dict(m))
print("enabled:", n.enabled_transitions(m))

{'Think0': 0, 'Fork0': 0, 'Eat0': 0, 'HasLeft0': 1, 'Think1': 0, 'Fork1': 0, 'Eat1': 0, 'HasLeft1': 1}
enabled: []


The same deadlock reproduced as a firing sequence: after `takeLeft0` and `takeLeft1` nothing is enabled.

In [5]:
n = examples.dining_philosophers(2, atomic=True)
s = space_of(n)
print(
    "deadlock free:",
    analysis.is_deadlock_free(s),
    "live:",
    analysis.is_live(s, set(n.transitions)),
    "safe:",
    analysis.is_safe(s),
)
for size in (2, 3, 4):
    naive = examples.dining_philosophers(size).reachable()
    fixed = examples.dining_philosophers(size, atomic=True).reachable()
    print(
        size,
        "naive has deadlock:",
        any(not x for x in naive.values()),
        "| atomic deadlock free:",
        all(fixed.values()),
    )

deadlock free: True live: True safe: True
2 naive has deadlock: True | atomic deadlock free: True
3 naive has deadlock: True | atomic deadlock free: True
4 naive has deadlock: True | atomic deadlock free: True


Taking both forks atomically removes the deadlock for every table size, while the naive variant always has one.

In [6]:
n = examples.mutex()
s = space_of(n)
print("states:", len(s))
print(
    "never both critical:",
    all(s.tokens(k, "Crit1") + s.tokens(k, "Crit2") <= 1 for k in s.graph),
)
print(
    "Crit1 reachable:", analysis.can_reach(s, lambda sp, k: sp.tokens(k, "Crit1") == 1)
)
print(
    "both reachable:",
    analysis.can_reach(
        s, lambda sp, k: sp.tokens(k, "Crit1") == 1 and sp.tokens(k, "Crit2") == 1
    ),
)
print("live:", analysis.is_live(s, set(n.transitions)))

states: 8
never both critical: True
Crit1 reachable: True
both reachable: False
live: True


Mutual exclusion holds in all 8 states; each process can enter its critical section, but never both together. The invariant `Mutex + Crit1 + Crit2 = 1` is the structural reason.

In [7]:
from petrilab.petri.model import PTNet

n = examples.mutex()
broken = PTNet("broken")
broken.places = dict(n.places)
for t, spec in n.transitions.items():
    broken.add_transition(
        t,
        {p: w for p, w in spec.pre.items() if p != "Mutex"},
        {p: w for p, w in spec.post.items() if p != "Mutex"},
    )
s = space_of(broken)
print(
    "violation reachable:",
    any(s.tokens(k, "Crit1") + s.tokens(k, "Crit2") == 2 for k in s.graph),
)

violation reachable: True


Removing the `Mutex` arcs lets both processes be critical at once, so the checker finds the safety violation.

In [8]:
n = examples.traffic_crossing()
s = space_of(n)
print("states:", len(s))
print(
    "never both green:",
    all(s.tokens(k, "NS_Green") + s.tokens(k, "EW_Green") <= 1 for k in s.graph),
)
print(
    "deadlock free:",
    analysis.is_deadlock_free(s),
    "live:",
    analysis.is_live(s, set(n.transitions)),
)
print(
    "Safe+greens invariant:",
    is_place_invariant(n, {"Safe": 1, "NS_Green": 1, "EW_Green": 1}),
)

states: 3
never both green: True
deadlock free: True live: True
Safe+greens invariant: True


The traffic crossing has 3 states (all red, NS green, EW green), conflicting greens are impossible thanks to the shared `Safe` token, and it is live.

In [9]:
for net in [
    examples.producer_consumer(3),
    examples.mutex(),
    examples.traffic_crossing(),
]:
    graph = net.reachable()
    invs = place_invariants(net)
    ok = all(
        len(
            {
                weighted_sum(net, dict(zip(net.place_names, y, strict=True)), m)
                for m in graph
            }
        )
        == 1
        for y in invs
    )
    print(net.name, "invariants:", len(invs), "constant on reachable markings:", ok)

producer_consumer_3 invariants: 3 constant on reachable markings: True
mutex invariants: 3 constant on reachable markings: True
traffic_crossing invariants: 3 constant on reachable markings: True


Every place invariant computed from the incidence matrix is constant over all reachable markings, as the theory predicts.